# 13 · Greedy Algorithms and Intervals

After this notebook you can decide whether a greedy choice is safe (and prove it with an exchange argument), recognise when it is not (and switch to DP), and solve the interval family — activity selection, merge, insert, meeting rooms, non-overlapping — plus jump game, gas station and the two knapsacks. You will also measure peak vs average concurrency of LLM calls, which is meeting rooms in production clothes.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Sorting algorithms](07_sorting_algorithms.ipynb), [Heaps and top-k](10_heaps_and_top_k.ipynb), [Dynamic programming](12_dynamic_programming.ipynb)

## Why this matters in interviews

- Greedy solutions are short (usually **sort, then one pass**) and fast — but only correct when a local choice can never hurt. Interviewers want the one-sentence argument, and a counterexample for the wrong greedy rule.
- Interval problems are a very common family; nearly all of them start with "sort by start" or "sort by end", and the choice between the two *is* the solution.
- "How many servers / slots / rooms do I need?" is meeting rooms II. In AI engineering that is the number of concurrent LLM calls to provision for — and the answer is set by the **peak**, not the average.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pd21` | How do you handle capacity planning for a spiky LLM workload? (peak vs average in-flight requests, measured from a request log) |
| `in29` | What is the difference between concurrency, batch size and throughput? (concurrency computed from request intervals) |

## What interviewers ask

- "Merge overlapping intervals" (LeetCode 56), "insert an interval" (57).
- "Can a person attend all meetings?" (252), "minimum number of meeting rooms" (253).
- "Minimum intervals to remove so the rest do not overlap" (435), "minimum arrows to burst balloons" (452).
- "Jump game I / II" (55 / 45), "gas station" (134), "partition labels" (763).
- "Maximum number of activities" (activity selection) and "fractional knapsack".
- Follow-ups: "prove greedy is optimal", "give a counterexample for the other sorting rule", "what if the intervals arrive as a stream?"

In [ ]:
import heapq
import itertools
import math
import random

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titlesize": 11,
                     "legend.frameon": False})
BLUE, ORANGE, AQUA, YELLOW, INK, MUTED, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#0b0b0b", "#52514e", "#c9c7c0"
rnd = random.Random(0)

def gantt(ax, intervals, rows=None, colors=None, labels=None):
    """Horizontal bars from start to end; rows default to one row per interval."""
    rows = list(range(len(intervals))) if rows is None else rows
    for k, ((s, e), r) in enumerate(zip(intervals, rows)):
        ax.barh(r, e - s, left=s, height=0.62, color=colors[k] if colors else BLUE, edgecolor="white", linewidth=1.5)
        if labels:
            ax.text((s + e) / 2, r, labels[k], ha="center", va="center", fontsize=8, color="white")
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)

## 1. When greedy works — the exchange argument

**In plain English:** a greedy algorithm makes the choice that looks best right now and never reconsiders. It is correct when you can show that *some* optimal answer starts with the greedy choice.

**The exchange argument** (the proof interviewers want, in words): "Take any optimal solution. If its first choice is not the greedy one, swap it for the greedy choice. The solution stays valid and is no worse. Repeat for the rest." If that swap can make things worse, greedy is wrong and you need DP.

**Template:**
```python
def greedy(items):
    result = []
    for item in sorted(items, key=THE_RIGHT_KEY):   # the whole trick is choosing this key
        if compatible(result, item):
            result.append(item)                   # commit, never undo
    return result
```

## 2. When greedy fails — coin change

"Always take the largest coin" is optimal for coin systems like `[1, 5, 10, 25]`, but not for `[1, 3, 4]`: for 6 it takes 4 + 1 + 1 (three coins) while 3 + 3 uses two. The exchange argument breaks: swapping a 4 into an optimal answer can force two extra coins.

In [ ]:
def greedy_coins(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count

def optimal_coins(coins, amount):                     # DP from notebook 12
    dp = [0] + [math.inf] * amount
    for a in range(1, amount + 1):
        dp[a] = 1 + min(dp[a - c] for c in coins if c <= a)
    return dp[amount]

amounts = list(range(1, 31))
g = [greedy_coins([1, 3, 4], a) for a in amounts]
o = [optimal_coins([1, 3, 4], a) for a in amounts]
worse = [a for a, x, y in zip(amounts, g, o) if x > y]
print("coins [1, 3, 4]: greedy is worse for amounts", worse)
assert greedy_coins([1, 3, 4], 6) == 3 and optimal_coins([1, 3, 4], 6) == 2
assert all(greedy_coins([1, 5, 10, 25], a) == optimal_coins([1, 5, 10, 25], a) for a in range(1, 201))
print("coins [1, 5, 10, 25]: greedy is optimal for every amount 1..200")

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(np.array(amounts) - 0.2, g, width=0.4, color=BLUE, label="greedy (largest coin first)")
ax.bar(np.array(amounts) + 0.2, o, width=0.4, color=ORANGE, label="optimal (DP)")
ax.set_xlabel("amount"); ax.set_ylabel("coins used"); ax.legend(loc="upper left")
ax.set_title("Coins [1, 3, 4]: wherever the blue bar is taller, the greedy choice was wrong")
plt.show()

## 3. Activity selection — the greedy that works

Given activities `(start, end)`, choose the **maximum number** that do not overlap (touching is fine). **Greedy:** sort by **end time**, take every activity that starts after the last one you took ended. **Why it is right:** the activity that ends first leaves the most room for everything else — exchange it into any optimal schedule and nothing breaks.

**Brute force:** try all 2^n subsets. And the tempting wrong rules — earliest start, shortest duration — each have a three-interval counterexample.

In [ ]:
def select_by(intervals, key):
    chosen, last_end = [], -math.inf
    for s, e in sorted(intervals, key=key):
        if s >= last_end:
            chosen.append((s, e)); last_end = e
    return chosen

by_end = lambda iv: iv[1]                            # the correct rule
by_start = lambda iv: iv[0]                          # wrong
by_length = lambda iv: iv[1] - iv[0]                 # wrong

def select_activities(intervals):
    return select_by(intervals, by_end)

def activities_brute(intervals):
    for r in range(len(intervals), -1, -1):
        for combo in itertools.combinations(sorted(intervals), r):
            if all(a[1] <= b[0] for a, b in zip(combo, combo[1:])):
                return r
    return 0

for _ in range(200):
    ivs = [(s, s + rnd.randint(1, 6)) for s in (rnd.randint(0, 20) for _ in range(rnd.randint(0, 10)))]
    assert len(select_activities(ivs)) == activities_brute(ivs)
print("earliest-end greedy == brute force on 200 random instances")

long_first = [(0, 10), (1, 2), (3, 4)]
short_middle = [(0, 5), (4, 7), (6, 11)]
print("earliest start on", long_first, "->", len(select_by(long_first, by_start)), "vs best", activities_brute(long_first))
print("shortest first on", short_middle, "->", len(select_by(short_middle, by_length)), "vs best", activities_brute(short_middle))
assert len(select_by(long_first, by_start)) == 1 and len(select_by(short_middle, by_length)) == 1
assert len(select_activities(long_first)) == len(select_activities(short_middle)) == 2

In [ ]:
arng = np.random.default_rng(4)
starts = arng.integers(0, 40, 14)
acts = sorted([(int(s), int(s + arng.integers(2, 9))) for s in starts], key=by_end)
chosen = set(select_activities(acts))
fig, ax = plt.subplots(figsize=(9, 3.8))
gantt(ax, acts, colors=[ORANGE if a in chosen else GREY for a in acts],
      labels=[f"{s}-{e}" for s, e in acts])
ax.set_yticks([]); ax.set_xlabel("time")
ax.set_title(f"Activities sorted by end time: greedy keeps the orange {len(chosen)} (the maximum)")
plt.show()
assert len(chosen) == activities_brute(acts)

**Complexity:** O(n log n) for the sort, O(1) extra space.

## 4. Merge intervals (LeetCode 56)

`[[1, 3], [2, 6], [8, 10], [15, 18]]` → `[[1, 6], [8, 10], [15, 18]]`. **Sort by start**; each interval either overlaps the last merged one (extend its end) or starts a new one. Touching intervals (`[1, 4]`, `[4, 5]`) merge.

**Brute force for checking:** paint every interval onto a number line (at doubled coordinates, so `[1, 2]` and `[3, 4]` stay separate) and read off the painted runs.

In [ ]:
def merge_intervals(intervals):
    merged = []
    for s, e in sorted(intervals):
        if merged and s <= merged[-1][1]:            # overlaps or touches the last one
            merged[-1][1] = max(merged[-1][1], e)    # max: the new one may be nested inside
        else:
            merged.append([s, e])
    return merged

def merge_brute(intervals):
    if not intervals:
        return []
    painted = set()
    for s, e in intervals:
        painted.update(range(2 * s, 2 * e + 1))
    runs, points = [], sorted(painted)
    start = prev = points[0]
    for p in points[1:]:
        if p != prev + 1:
            runs.append([start // 2, prev // 2]); start = p
        prev = p
    runs.append([start // 2, prev // 2])
    return runs

assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]] and merge_intervals([[1, 10], [2, 3]]) == [[1, 10]]
assert merge_intervals([]) == []
for _ in range(300):
    ivs = [[s, s + rnd.randint(0, 5)] for s in (rnd.randint(0, 30) for _ in range(rnd.randint(0, 10)))]
    assert merge_intervals(ivs) == merge_brute(ivs)
print("merge == number-line painting on 300 random inputs")

before = [[1, 3], [2, 6], [8, 10], [9, 12], [15, 18], [16, 17]]
after = merge_intervals(before)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), sharex=True)
gantt(axes[0], before, colors=[BLUE] * len(before)); axes[0].set_title("input, sorted by start"); axes[0].set_yticks([])
gantt(axes[1], after, colors=[ORANGE] * len(after)); axes[1].set_title(f"merged: {after}"); axes[1].set_yticks([])
plt.show()

**Complexity:** O(n log n) time, O(n) for the output.

**GenAI tie-in — merging redaction spans.** PII detectors (a regex for emails, an NER model for names, a rule for phone numbers) return **overlapping** character spans. Merge them before masking, or overlapping spans get masked twice and the offsets of later spans go wrong.

In [ ]:
text = "Contact Jane Doe at jane.doe@example.com or +1 555 0100 today."
spans = [(8, 16),        # NER: "Jane Doe"
         (20, 40),       # regex: the email address
         (20, 28),       # NER: "jane.doe" inside the email
         (44, 55)]       # phone number
merged_spans = merge_intervals([list(sp) for sp in spans])
redacted, cursor = [], 0
for s, e in merged_spans:
    redacted.append(text[cursor:s] + "[REDACTED]"); cursor = e
redacted = "".join(redacted) + text[cursor:]
print("merged spans:", merged_spans)
print(redacted)
assert "jane" not in redacted.lower() and "0100" not in redacted and redacted.count("[REDACTED]") == 3

## 5. Insert interval (LeetCode 57)

The intervals are already sorted and non-overlapping; insert one more. One pass in three phases: copy everything that **ends before** the new one starts, **absorb** everything that overlaps it, copy the rest. O(n) — no sort needed.

In [ ]:
def insert_interval(intervals, new):
    out, i, n = [], 0, len(intervals)
    s, e = new
    while i < n and intervals[i][1] < s:            # phase 1: entirely before
        out.append(list(intervals[i])); i += 1
    while i < n and intervals[i][0] <= e:           # phase 2: overlapping -> absorb
        s, e = min(s, intervals[i][0]), max(e, intervals[i][1]); i += 1
    out.append([s, e])
    out.extend(list(iv) for iv in intervals[i:])    # phase 3: entirely after
    return out

assert insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [[1, 2], [3, 10], [12, 16]]
assert insert_interval([], [5, 7]) == [[5, 7]] and insert_interval([[1, 5]], [6, 8]) == [[1, 5], [6, 8]]
for _ in range(300):
    base = merge_intervals([[s, s + rnd.randint(0, 4)] for s in (rnd.randint(0, 40) for _ in range(rnd.randint(0, 8)))])
    s = rnd.randint(0, 45)
    new = [s, s + rnd.randint(0, 8)]
    assert insert_interval(base, new) == merge_intervals(base + [new])
print(insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))

## 6. Meeting rooms II (LeetCode 253) — minimum rooms with a heap

Sort meetings by start. Keep a **min-heap of end times** of rooms in use. For each meeting: if the room that frees up first is free by now, reuse it; otherwise open a new room. The number of rooms opened is the answer — and equals the **maximum number of meetings in progress at any moment**.

**Sweep-line alternative:** turn every meeting into a `+1` event at its start and a `−1` at its end, sort (ends before starts at the same time), and track the running count. **Brute force:** at every start time, count the meetings in progress, O(n²).

In [ ]:
def assign_rooms(meetings):
    """Greedy room assignment. Returns (rooms_needed, room_of_each_meeting)."""
    free_at, rooms, opened = [], [None] * len(meetings), 0      # free_at: heap of (end, room)
    for i in sorted(range(len(meetings)), key=lambda i: meetings[i][0]):
        s, e = meetings[i]
        if free_at and free_at[0][0] <= s:
            _, room = heapq.heappop(free_at)                     # reuse the room that frees up first
        else:
            room, opened = opened, opened + 1                    # all busy: open a new one
        rooms[i] = room
        heapq.heappush(free_at, (e, room))
    return opened, rooms

def rooms_sweep(meetings):
    events = sorted([(s, 1) for s, _ in meetings] + [(e, -1) for _, e in meetings])   # (-1) sorts first on ties
    current = peak = 0
    for _, delta in events:
        current += delta
        peak = max(peak, current)
    return peak

def rooms_brute(meetings):
    return max((sum(s <= t < e for s, e in meetings) for t, _ in meetings), default=0)

assert assign_rooms([(0, 30), (5, 10), (15, 20)])[0] == 2 == rooms_sweep([(0, 30), (5, 10), (15, 20)])
assert assign_rooms([(7, 10), (2, 4)])[0] == 1 and assign_rooms([])[0] == 0
assert rooms_sweep([(1, 5), (5, 10)]) == 1                       # back-to-back meetings share a room
for _ in range(300):
    ms = [(s, s + rnd.randint(1, 10)) for s in (rnd.randint(0, 30) for _ in range(rnd.randint(0, 12)))]
    assert assign_rooms(ms)[0] == rooms_sweep(ms) == rooms_brute(ms)
print("heap == sweep line == brute force on 300 random schedules")

In [ ]:
mrng = np.random.default_rng(7)
meet = [(int(s), int(s + mrng.integers(2, 8))) for s in np.sort(mrng.integers(0, 24, 12))]
n_rooms, room_of = assign_rooms(meet)
fig, ax = plt.subplots(figsize=(9, 3))
gantt(ax, meet, rows=room_of, colors=[BLUE] * len(meet), labels=[f"m{i}" for i in range(len(meet))])
ax.set_yticks(range(n_rooms), [f"room {r}" for r in range(n_rooms)]); ax.set_xlabel("time")
ax.set_title(f"12 meetings packed into {n_rooms} rooms: a new room opens only when every room is busy")
plt.show()
assert n_rooms == rooms_brute(meet)

**Complexity:** O(n log n) time, O(n) space.

### GenAI tie-in: how many LLM calls are in flight? (`pd21`, `in29`)

Replace "meetings" with "LLM requests" and "rooms" with "concurrency slots" (provider rate-limit budget, worker processes, GPU batch slots). The sweep line over a request log gives the concurrency at every instant. **Little's law** gives the average: *average in flight = arrival rate × average duration*. Capacity has to cover the **peak** — the average hides the burst.

In [ ]:
crng = np.random.default_rng(0)
window = 600.0                                                  # ten minutes of traffic, in seconds
steady = crng.uniform(0, window, 400)                           # background traffic
burst = crng.uniform(300, 330, 150)                             # a 30-second spike (a batch job, a launch)
starts = np.sort(np.concatenate([steady, burst]))
durations = crng.lognormal(mean=1.0, sigma=0.6, size=len(starts))   # seconds per LLM call
requests = list(zip(starts, starts + durations))

peak = rooms_sweep(requests)
average = durations.sum() / window                              # Little's law: total busy time / window
print(f"{len(requests)} requests | mean call {durations.mean():.2f} s | "
      f"average in flight {average:.1f} | peak in flight {peak} | peak-to-average {peak / average:.1f}x")
assert peak == assign_rooms(requests)[0] and peak > 3 * average

events = sorted([(s, 1) for s, _ in requests] + [(e, -1) for _, e in requests])
times, level, cur = [], [], 0
for t, delta in events:
    cur += delta; times.append(t); level.append(cur)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.step(times, level, where="post", color=BLUE, lw=1.4)
ax.axhline(average, color=MUTED, lw=1.5)
ax.axhline(peak, color=ORANGE, lw=1.5)
ax.text(5, average + peak * 0.03, f"average {average:.1f}", color=MUTED, fontsize=9)
ax.text(5, peak + peak * 0.03, f"peak {peak}", color=ORANGE, fontsize=9)
ax.set_xlabel("time (s)"); ax.set_ylabel("LLM calls in flight"); ax.set_ylim(0, peak * 1.15)
ax.set_title("Concurrency from a request log (sweep line): provision for the peak, not the average")
plt.show()

If you provision for the average, the burst either queues (latency spikes) or gets 429s. The production answers — a bounded queue, priorities, shedding low-value work, a cheaper model during the spike — are covered in [async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb) and [retries, backoff and rate limits](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb).

## 7. Non-overlapping intervals (LeetCode 435)

Remove the fewest intervals so the rest do not overlap. That is "keep the most", i.e. activity selection: `answer = n − (maximum non-overlapping set)`, greedy by earliest end.

In [ ]:
def erase_overlap_intervals(intervals):
    return len(intervals) - len(select_activities(intervals))

assert erase_overlap_intervals([(1, 2), (2, 3), (3, 4), (1, 3)]) == 1
assert erase_overlap_intervals([(1, 2), (1, 2), (1, 2)]) == 2
assert erase_overlap_intervals([(1, 2), (2, 3)]) == 0 and erase_overlap_intervals([]) == 0
print("remove from [(1,2),(2,3),(3,4),(1,3)]:", erase_overlap_intervals([(1, 2), (2, 3), (3, 4), (1, 3)]))

## 8. Jump game (LeetCode 55)

`nums[i]` is the maximum jump length from index `i`. Can you reach the last index? **Brute force / DP:** mark every index reachable from each reachable index, O(n²). **Greedy:** track the farthest index reachable so far; if you ever stand beyond it, you are stuck. O(n).

In [ ]:
def can_jump(nums):
    farthest = 0
    for i, jump in enumerate(nums):
        if i > farthest:
            return False                         # index i can never be reached
        farthest = max(farthest, i + jump)
    return True

def can_jump_dp(nums):
    ok = [False] * len(nums)
    ok[0] = True
    for i, jump in enumerate(nums):
        if ok[i]:
            for j in range(i + 1, min(len(nums), i + jump + 1)):
                ok[j] = True
    return ok[-1]

assert can_jump([2, 3, 1, 1, 4]) and not can_jump([3, 2, 1, 0, 4]) and can_jump([0])
for _ in range(300):
    nums = [rnd.randint(0, 3) for _ in range(rnd.randint(1, 15))]
    assert can_jump(nums) == can_jump_dp(nums)
print("[2,3,1,1,4] ->", can_jump([2, 3, 1, 1, 4]), "| [3,2,1,0,4] ->", can_jump([3, 2, 1, 0, 4]))

## 9. Gas station (LeetCode 134)

Stations on a circle; `gas[i]` fuel at station `i`, `cost[i]` to drive to the next. Find the start that completes the loop, or −1. **Brute force:** simulate from every start, O(n²). **Greedy, two facts:** (1) if total gas < total cost, no start works; (2) if the tank goes negative on the way from `start` to `i`, no station between them can be the start either — jump the start to `i + 1`. O(n).

**Picture:** plot the running balance starting from station 0. The answer is the station where that balance is **lowest**: start there and the tank is never below its starting level.

In [ ]:
def can_complete_circuit(gas, cost):
    if sum(gas) < sum(cost):
        return -1
    start, tank = 0, 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:
            start, tank = i + 1, 0                 # nothing in [start, i] can work
    return start

def circuit_brute(gas, cost):
    n = len(gas)
    for s in range(n):
        tank = 0
        for k in range(n):
            i = (s + k) % n
            tank += gas[i] - cost[i]
            if tank < 0:
                break
        else:
            return s
    return -1

assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert can_complete_circuit([2, 3, 4], [3, 4, 3]) == -1
for _ in range(300):
    n = rnd.randint(1, 10)
    gas, cost = [rnd.randint(0, 6) for _ in range(n)], [rnd.randint(0, 6) for _ in range(n)]
    assert can_complete_circuit(gas, cost) == circuit_brute(gas, cost)
print("greedy == simulate-every-start on 300 random circuits")

In [ ]:
srng = np.random.default_rng(2)
gas = srng.integers(1, 9, 12)
cost = srng.integers(1, 9, 12)
gas[0] += max(0, int(cost.sum() - gas.sum()))                  # make the loop feasible
balance = np.concatenate([[0], np.cumsum(gas - cost)])[:-1]    # tank on arrival at station i, starting at 0
start = can_complete_circuit(gas.tolist(), cost.tolist())
assert start == int(np.argmin(balance)) and start == circuit_brute(gas.tolist(), cost.tolist())

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(range(12), balance, color=BLUE, lw=2, marker="o", ms=5)
ax.scatter([start], [balance[start]], s=120, color=ORANGE, zorder=3)
ax.annotate(f"lowest point: start at station {start}", (start, balance[start]), xytext=(10, -18),
            textcoords="offset points", color=ORANGE, fontsize=9)
ax.axhline(0, color=MUTED, lw=1)
ax.set_xticks(range(12)); ax.set_xlabel("station"); ax.set_ylabel("fuel on arrival (from station 0)")
ax.set_title("Gas station: start where the running balance bottoms out")
plt.show()

## 10. Fractional vs 0/1 knapsack

With a **fractional** knapsack you may take part of an item, so "best value per kilogram first" is optimal — any leftover capacity is filled with the next-best ratio. With **0/1** items the same rule fails: a high-ratio small item can waste capacity that two larger items would have filled exactly. The classic example: items `(value, weight)` = `(60, 10), (100, 20), (120, 30)`, capacity 50.

In [ ]:
def fractional_knapsack(items, cap):
    total = 0.0
    for value, weight in sorted(items, key=lambda it: it[0] / it[1], reverse=True):
        take = min(weight, cap)
        total += value * take / weight
        cap -= take
        if cap == 0:
            break
    return total

def greedy_01(items, cap):                                   # the same ratio rule, whole items only
    total = 0
    for value, weight in sorted(items, key=lambda it: it[0] / it[1], reverse=True):
        if weight <= cap:
            total += value; cap -= weight
    return total

def knapsack_01(items, cap):                                 # DP from notebook 12
    dp = [0] * (cap + 1)
    for value, weight in items:
        for w in range(cap, weight - 1, -1):
            dp[w] = max(dp[w], dp[w - weight] + value)
    return dp[cap]

items = [(60, 10), (100, 20), (120, 30)]
print(f"fractional greedy: {fractional_knapsack(items, 50):.0f} | 0/1 greedy by ratio: {greedy_01(items, 50)} "
      f"| 0/1 optimal (DP): {knapsack_01(items, 50)}")
assert (fractional_knapsack(items, 50), greedy_01(items, 50), knapsack_01(items, 50)) == (240, 160, 220)

| Problem | Greedy rule | Greedy optimal? | Otherwise |
|---|---|---|---|
| activity selection / non-overlapping | earliest end | yes | — |
| merge / insert intervals | sort by start, sweep | yes | — |
| meeting rooms | sort by start, heap of ends | yes | — |
| fractional knapsack | best value per weight | yes | — |
| 0/1 knapsack | best value per weight | **no** | DP, O(n · W) |
| coin change | largest coin first | only for canonical coin systems | DP, O(amount · coins) |
| jump game / gas station | farthest reach / reset on negative | yes | — |

## Try it yourself

**1. Minimum arrows to burst balloons (LeetCode 452).** Each balloon spans `[start, end]`; an arrow at `x` bursts every balloon with `start ≤ x ≤ end`. Hint: sort by end and shoot at the end of the first balloon not yet burst.

In [ ]:
# Solution — try it yourself first, then run this
def min_arrows(balloons):
    arrows, last = 0, -math.inf
    for s, e in sorted(balloons, key=lambda b: b[1]):
        if s > last:                          # this balloon is not burst yet
            arrows += 1
            last = e                          # shoot at its end: bursts every balloon covering e
    return arrows

assert min_arrows([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2
assert min_arrows([[1, 2], [3, 4], [5, 6], [7, 8]]) == 4
assert min_arrows([[1, 2], [2, 3], [3, 4], [4, 5]]) == 2 and min_arrows([]) == 0
print(min_arrows([[10, 16], [2, 8], [1, 6], [7, 12]]))

**2. Jump game II (LeetCode 45).** Minimum number of jumps to reach the last index (always reachable). Hint: BFS by levels without a queue — the indices reachable with `j` jumps form a window; the next window ends at the farthest point reachable from it.

In [ ]:
# Solution — try it yourself first, then run this
def min_jumps(nums):
    jumps, window_end, farthest = 0, 0, 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == window_end:                   # finished this level: one more jump
            jumps += 1
            window_end = farthest
    return jumps

def min_jumps_dp(nums):
    best = [0] + [math.inf] * (len(nums) - 1)
    for i in range(len(nums)):
        for j in range(i + 1, min(len(nums), i + nums[i] + 1)):
            best[j] = min(best[j], best[i] + 1)
    return best[-1]

assert min_jumps([2, 3, 1, 1, 4]) == 2 and min_jumps([2, 3, 0, 1, 4]) == 2 and min_jumps([0]) == 0
for _ in range(300):
    nums = [rnd.randint(1, 4) for _ in range(rnd.randint(1, 15))]
    assert min_jumps(nums) == min_jumps_dp(nums)
print(min_jumps([2, 3, 1, 1, 4]))

**3. Partition labels (LeetCode 763).** Split a string into as many parts as possible so each letter appears in at most one part; return the part sizes. Hint: it is merging intervals — each letter spans from its first to its last occurrence.

In [ ]:
# Solution — try it yourself first, then run this
def partition_labels(s):
    last = {ch: i for i, ch in enumerate(s)}
    sizes, start, end = [], 0, 0
    for i, ch in enumerate(s):
        end = max(end, last[ch])              # this part must reach the last copy of ch
        if i == end:
            sizes.append(end - start + 1)
            start = i + 1
    return sizes

assert partition_labels("ababcbacadefegdehijhklij") == [9, 7, 8]
assert partition_labels("eccbbbbdec") == [10] and partition_labels("abc") == [1, 1, 1]
print(partition_labels("ababcbacadefegdehijhklij"))

## Say it in an interview

**30-second answer:** "Greedy commits to the locally best choice and never undoes it. I only use it when I can argue that some optimal answer starts with that choice — the exchange argument — and I test the rule against a small counterexample. For intervals I sort: by end time when I want to keep as many as possible (activity selection, non-overlapping, arrows), by start time when I merge or allocate (merge, insert, meeting rooms with a min-heap of end times). Everything is O(n log n) for the sort plus one pass."

**On capacity (`pd21`):** "Meeting rooms II over a request log gives the peak number of in-flight LLM calls. Little's law gives the average: arrival rate × mean latency. Provision for the peak, or put a bounded queue with priorities in front and decide what to shed."

**Follow-ups and traps:**
- *Sort key:* earliest end for "keep the most"; earliest start or shortest duration have counterexamples.
- *Touching intervals:* decide `<` vs `<=` from the problem statement (merge touches; meetings that end at 10 free the room for one starting at 10).
- *Nested intervals:* merging needs `max(end, new_end)`, not `new_end`.
- *Greedy vs DP:* coin change with `[1, 3, 4]` and 0/1 knapsack are the standard counterexamples.
- *Sweep line tie-break:* process ends before starts at the same timestamp.

## Pattern cheat sheet

| Signal | Recipe | Time |
|---|---|---|
| keep the most non-overlapping / fewest removals / fewest arrows | sort by **end**, take if `start >= last_end` | O(n log n) |
| merge / union of intervals | sort by **start**, extend `merged[-1]` | O(n log n) |
| insert into sorted intervals | three phases: before, absorb, after | O(n) |
| rooms / servers / concurrency | min-heap of end times, or +1/−1 sweep line | O(n log n) |
| can I reach the end? | track farthest reach | O(n) |
| minimum jumps | BFS by windows | O(n) |
| circular fuel / balance | total check + reset start on negative | O(n) |
| take fractions by value density | sort by value/weight | O(n log n) |
| greedy fails the exchange argument | switch to DP ([notebook 12](12_dynamic_programming.ipynb)) | — |

## Next

- [10 · Heaps and top-k](10_heaps_and_top_k.ipynb) — the heap behind meeting rooms and task scheduling.
- [12 · Dynamic programming](12_dynamic_programming.ipynb) — for when greedy fails.
- [Async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb) · [Latency percentiles p50/p95 and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [Guardrails: PII and prompt injection](../16_production/07_guardrails_pii_and_prompt_injection.ipynb)
- Theory: [DSA basics course — algorithm paradigms chapter](../../dsa_basics/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)